# Course 10 lab — From specification to agent work units

This credential-free notebook uses fictional, versioned fixtures. It evaluates planning relationships; it does not authorize agents, inspect a live repository, or prove implementation correctness.

In [ ]:
import copy
import importlib.util
import sys
from pathlib import Path
spec = importlib.util.spec_from_file_location('course10_notebook_lab', Path.cwd() / 'lab.py')
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
WorkUnitState = lab.WorkUnitState
assess_plan_staleness = lab.assess_plan_staleness
coordination_metrics = lab.coordination_metrics
critical_path = lab.critical_path
disposition_metrics = lab.disposition_metrics
downstream_work_units = lab.downstream_work_units
evaluate_planning_rules = lab.evaluate_planning_rules
generate_execution_context = lab.generate_execution_context
implementation_traceability = lab.implementation_traceability
load_bundle = lab.load_bundle
planning_decision = lab.planning_decision
ready_work_units = lab.ready_work_units
review_plan = lab.review_plan
topological_waves = lab.topological_waves
transition_work_unit_state = lab.transition_work_unit_state
validate_completion_report = lab.validate_completion_report
validate_contract_dependencies = lab.validate_contract_dependencies
validate_contract_change_request = lab.validate_contract_change_request
reference = load_bundle()
candidate = load_bundle(candidate=True)

## 1. Diagnose ticket-shaped planning

The candidate jumps from a ticket to broad agent work. Inspect typed findings instead of collapsing them into a risk score.

In [ ]:
candidate_report = review_plan(candidate)
candidate_decision = planning_decision(candidate_report)
candidate_decision['state'], candidate_report['counts'], candidate_decision['finding_codes'][:8]

The candidate must stop. Architecture invention, protected writes, a blocked capability, and missing evidence cannot be averaged away.

## 2. Review the governed reference

In [ ]:
reference_report = review_plan(reference)
planning_decision(reference_report), reference_report['findings']

In [ ]:
disposition_metrics(reference['plan'], reference['specification']), implementation_traceability(reference['plan'], reference['specification'])['complete_requirement_chains']

Coverage is structural. A complete chain makes omissions visible but does not prove that the plan is correct.

## 3. Schedule dependency-aware waves

In [ ]:
topological_waves(reference['plan']), coordination_metrics(reference['plan'])

In [ ]:
ready_work_units(reference['plan']), ready_work_units(reference['plan'], completed_work_unit_ids=['AWU-BR-CONTRACT'])

Extraction and validation become ready together only after the shared contract unit completes. Readiness also depends on current linked evidence.

## 4. Generate bounded execution context

In [ ]:
context = generate_execution_context(reference, 'AWU-BR-EXTRACTION')
{key: context[key] for key in ('work_unit_id', 'plan_digest', 'repository_revision', 'writable_paths', 'protected_decisions', 'accountability_team', 'execution_owner', 'authority_boundary')}

The context carries constraints and provenance. It deliberately carries no credential, production permission, or policy exception.

## 5. Distinguish relevant from unrelated drift

In [ ]:
unrelated_repo = copy.deepcopy(reference['repository'])
unrelated_repo['revision'] = 'repo-def456'
unrelated_repo['changed_paths_since_plan'] = ['docs/style.md']
relevant_repo = copy.deepcopy(unrelated_repo)
relevant_repo['changed_paths_since_plan'] = ['src/underwriting/proposals.py']
assess_plan_staleness(reference['plan'], unrelated_repo), assess_plan_staleness(reference['plan'], relevant_repo)

## 6. Propagate a contract change

In [ ]:
affected = downstream_work_units(reference['plan'], 'AWU-BR-EXTRACTION')
request = {
    'id': 'CCR-2219-01', 'discovered_by': 'AWU-BR-EXTRACTION',
    'contract': 'ProposedUpdate@2', 'proposed_change': 'add confidence_reason',
    'reason_requirement_ids': ['REQ-BR-030'], 'affected_work_unit_ids': affected,
    'impact': {'compatibility': 'review', 'evidence': 'rerun downstream'},
}
affected, validate_contract_change_request(request, reference['plan'])

A downstream discovery becomes a proposal and impact record, not an unauthorized upstream edit.

## 7. Assure contract dependencies, stop routing, and permissions

Explicit dependency edges are compared with the authoritative contract registry. The critical path uses coarse work indicators, not elapsed-time promises.

In [ ]:
validate_contract_dependencies(reference['plan']), critical_path(reference['plan'])

In [ ]:
stop_routes = {item['condition']: (item['outcome'], item['required_artifact'], item['owner']) for item in reference['plan']['stop_condition_catalog']}
permission = next(item for item in reference['plan']['permission_profiles'] if item['work_unit_id'] == 'AWU-BR-EXTRACTION')
stop_routes, permission

Every consequential discovery routes to `ASK`, `PROPOSE`, or `STOP` with a durable artifact and accountable owner. The permission object remains a temporary request in `planned_not_provisioned`; neither planner nor agent can grant it.

## 8. Separate implementation readiness from verification readiness

In [ ]:
verification_delayed = copy.deepcopy(reference['plan'])
next(item for item in verification_delayed['readiness_evidence_catalog'] if item['id'] == 'VERIFICATION-INFRA-2219')['status'] = 'missing'
normal_readiness = ready_work_units(reference['plan'])
delayed_readiness = ready_work_units(verification_delayed)
normal_readiness, delayed_readiness

The contract work unit can remain implementation-ready while independent verification is blocked. That state must never be presented as fully verified, merge-ready, or enablement-ready.

## 9. Check path scope and semantic scope

In [ ]:
unit = next(item for item in reference['plan']['work_units'] if item['id'] == 'AWU-BR-EXTRACTION')
bad_completion = {
    'status': 'COMPLETED', 'evidence_ids': [],
    'semantic_actions': ['mutation_eligibility'], 'unresolved': ['contract ambiguity'],
}
[finding.code for finding in validate_completion_report(unit, bad_completion, actual_changed_paths=['src/underwriting/submission.py'])]

An allowed path set is necessary but insufficient: protected semantics must also be checked.

## 10. Enforce lifecycle evidence gates

In [ ]:
state = transition_work_unit_state(WorkUnitState.PROPOSED, WorkUnitState.READY)
state = transition_work_unit_state(state, WorkUnitState.IN_PROGRESS)
state = transition_work_unit_state(state, WorkUnitState.COMPLETED)
state = transition_work_unit_state(state, WorkUnitState.VERIFIED, verification_passed=True)
state = transition_work_unit_state(state, WorkUnitState.INTEGRATED, integration_passed=True)
state

## 11. Evaluate disclosed rule coverage

In [ ]:
evaluation = evaluate_planning_rules()
{key: evaluation[key] for key in ('claim', 'population', 'true_positive', 'false_positive', 'false_negative', 'exact_matches')}

Thirty exact matches establish regression behavior on this labelled fixture only. They do not establish planner accuracy on undisclosed repositories.

## Production upgrade

| Fixture mechanism | Production upgrade |
|---|---|
| JSON evidence IDs | Authenticated, immutable attestations bound to exact revisions |
| Planned permission profile | External control-plane approval, short-lived credential, and atomic revocation |
| In-memory DAG | Durable scheduler with leases, cancellation, restart, and concurrency control |
| Contract registry | Published schema provenance, compatibility checks, and cross-repository release ordering |
| Labelled rule cases | Independently labelled cross-domain evaluation and production feedback |
| Rollout boundary | Authorized deployment, shadow, cohort, monitoring, rollback, and enablement controls |

## Reflection

1. Which candidate finding requires product clarification, architecture review, scope expansion, or local repair?
2. Which two reference units can run in parallel, and what makes that safe?
3. What evidence would you authenticate in a production system?
4. Which small change in your organization should use a lighter process?